# Problem Statement
**Problem statement:** Can we create a model that forecasts a week of covid vaccination rate with a 90% predicition accuracy?

**Background:** COVID-19 is an emergent disease that had no historical data to guide predicting/forecasting over time.

**Context/situation:** The ability to predict the progress of the vaccination rate is crucial to decision making aimed at reopening governments, businesses, and borders.

**Objectives:** To forecast the covid vaccine rate with a 90% prediction accuracy (p-values and uncertainty come into play here, will adjust my objective once I know more the importance of these statistical metrics with regards to time series modeling)

**Models (potentially):**
Recurrent Neural Network (RNN)
Autoregressive integrated moving average (ARIMA)
Autoregressive Poision model (ARPOIS)
cubic smoothing spline
NAIVE
Holt model – linear trend method, and winter seasonal method
Trigonometric Exponential smoothing state space model with Box-Cox transformation (TBATS)

**Metrics:** these are metrics for ARIMA, still need to look into others
ME
MAE
MSE
MPE (mean percentage error)
MAPE (mean absolute percentage error)
MASE (mean absolute scaled error)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import missingno as msno

from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GRU, SimpleRNN
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

In [ ]:
df = pd.read_csv('../datasets/us_state_vaccinations.csv')

In [ ]:
df.head()

In [ ]:
df.set_index('date', inplace=True)

In [ ]:
df.sort_index(axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df.index = pd.to_datetime(df.index)

In [ ]:
df.info()

In [ ]:
# creating a mask to set the dataframe only equal to the state of massachusetts
mask = df['location'] == 'Massachusetts'
df = df[mask]

In [ ]:
df.isnull().sum()

In [ ]:
df.head(20)

Only missing data through 7 rows (index 6)

In [ ]:
df.shape

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

In [ ]:
df.index.unique()

jan 12, 14 16 17 18 were dropped
Feb 15 was dropped
Should I just drop the first several rows then? I think it'll make the data confusing.
Also, need to look into feb 15, and probably impute on that row 

In [ ]:
# dropping the location column
df.drop('location', axis=1, inplace=True)
df.head()

In [ ]:
df.shape

## Plotting my data!

In [ ]:
df.columns

In [ ]:
df['total_vaccinations_per_hundred'].plot();

In [ ]:
df['people_fully_vaccinated'].plot();

In [ ]:
df['people_vaccinated'].plot();

In [ ]:
plt.plot(df['daily_vaccinations']);

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Code written by Joseph Nelson.

def interpret_dftest(dftest):
    dfoutput = pd.Series(dftest[0:2], index=['Test Statistic','p-value'])
    return dfoutput

# Run ADF test on original (non-differenced!) data.
interpret_dftest(adfuller(df['daily_vaccinations']))

In [ ]:
#sooo definitelly not stationary

In [ ]:
# df = df.diff()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
# Run ADF test on differenced data.
interpret_dftest(adfuller(df['daily_vaccinations']))

In [ ]:
# Run ADF test on differenced data.
interpret_dftest(adfuller(df['target']))

In [ ]:
# Run ADF test on differenced data.
interpret_dftest(adfuller(df['daily_vaccinations_per_million']))

In [ ]:
plt.figure(figsize=(12,12))
plt.plot(df['daily_vaccinations']);

In [ ]:
df.rename(columns={'daily_vaccinations_raw': 'target'}, inplace=True)

In [35]:
features = ['daily_vaccinations', 'daily_vaccinations_per_million']

In [36]:
X = df[features]
y = df.target.values

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False)
X_train.head(3)

,daily_vaccinations,daily_vaccinations_per_million
date,,
2021-01-15,4337.0,630.0
2021-01-19,-1299.0,-189.0
2021-01-20,-249.0,-36.0


In [41]:
# scale
ss = StandardScaler()
ss.fit(X_train)
X_train_sc = ss.transform(X_train)
X_test_sc = ss.transform(X_test)

In [57]:
len(X_train_sc)

66

In [58]:
len(y_train)

66

In [76]:
# Create training sequences
train_sequences = TimeseriesGenerator(data=X_train_sc,
                                      targets=y_train,
                                      length=1,
                                      batch_size=16)

In [77]:
# Create test sequences
test_sequences = TimeseriesGenerator(data=X_test_sc,
                                    targets=y_test,
                                    length=1,
                                    batch_size=16)

In [78]:
# Design RNN
train_sequences[0][0].shape, test_sequences[0][1].shape

((16, 1, 2), (16,))

In [72]:
# input shape setting up
input_shape = train_sequences[0][0][0].shape

# starting the model
model = Sequential()

# input - GRU! - needs a specified input_shape
model.add(GRU(8, activation = 'relu', recurrent_activation='relu', input_shape=input_shape, return_sequences=True))

# # middle layer - Dense
# model.add(Dense(1, activation = 'relu'))

# last layer - Dense
model.add(Dense(1))

In [ ]:
# # input shape setting up
# input_shape = train_sequences[0][0][0].shape

# # starting the model
# model = Sequential()

# # input - GRU! - needs a specified input_shape
# model.add(GRU(8, input_shape=input_shape, return_sequences=True))

# # next layer - also GRU! - next will be dense...
# model.add(GRU(8, return_sequences=False))

# # last layer - Dense
# model.add(Dense(1, activation='relu'))

In [73]:
# compile model
model.compile(optimizer=Adam(lr=.0005), loss='mse', metrics=['acc'])

In [75]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
gru_3 (GRU)                  (None, 1, 8)              288       
_________________________________________________________________
dense_3 (Dense)              (None, 1, 1)              9         
Total params: 297
Trainable params: 297
Non-trainable params: 0
_________________________________________________________________


In [74]:
history = model.fit(train_sequences, validation_data=test_sequences,
                   epochs=100, verbose=1)

Epoch 1/100
65/65 [==============================] - 2s 9ms/step - loss: 450380957.3333 - acc: 0.0000e+00 - val_loss: 458958784.0000 - val_acc: 0.0000e+00
Epoch 2/100
65/65 [==============================] - 0s 4ms/step - loss: 691983397.0909 - acc: 0.0000e+00 - val_loss: 458958752.0000 - val_acc: 0.0000e+00
Epoch 3/100
65/65 [==============================] - 0s 3ms/step - loss: 674083324.7273 - acc: 0.0000e+00 - val_loss: 458958656.0000 - val_acc: 0.0000e+00
Epoch 4/100
65/65 [==============================] - 0s 4ms/step - loss: 607923345.4545 - acc: 0.0000e+00 - val_loss: 458958560.0000 - val_acc: 0.0000e+00
Epoch 5/100
65/65 [==============================] - 0s 4ms/step - loss: 758650503.7576 - acc: 0.0000e+00 - val_loss: 458958464.0000 - val_acc: 0.0000e+00
Epoch 6/100
65/65 [==============================] - 0s 3ms/step - loss: 572683329.6970 - acc: 0.0000e+00 - val_loss: 458958400.0000 - val_acc: 0.0000e+00
Epoch 7/100
65/65 [==============================] - 0s 3ms/step - los